# Notebook 5 — The model against CodeScene

Chapter 5's model is compared with **CodeScene's Code Health** on the **same test rows**: the same files at the
same test releases, with the same labels. Code Health is CodeScene's 1–10 score of how maintainable a file's code
is (10 healthy, 1 severe problems). CodeScene is not
trained, so it needs no training or validation releases. It only scores each test file **as the file was at its
release**, which is what the model saw too.

| method | its score for a file at a release | from |
|---|---|---|
| the model | the probability it gave the file in run 3 | `results/run3/outputs/forecasts.parquet` |
| Code Health | 10 − Code Health of the file at the release's snapshot commit (unhealthiest first) | `cs review` |

Both are scored by **AP** and **ROC-AUC**. These depend only on the order the scores put the files in, not on the
scores' values, so a probability and a health score are compared on equal terms. Each is computed **within each
repo** and then averaged over the five repos, because either tool is used by a repo's developers to rank that
repo's files; neither ranks files of different repos against each other.

**Runs anywhere Jupyter runs**: local Jupyter, VS Code or Colab, on Linux, macOS or Windows. It needs Python with
`numpy`, `pandas`, `pyarrow` and `scikit-learn` (`pip install -r requirements.txt`; Colab has them), `git` and
the CodeScene CLI (installed by the notebook if missing), signed in with a free CodeScene account's Personal
Access Token, which the notebook asks for (section 1).

It uses the workspace the other notebooks share: `QUARRY_HOME` if set, otherwise `workspace_data/` one level
above the notebook (run it from `notebooks/`). From it, it needs each repo's **release grid**
(`versions/rel_hist_<repo>.json`, which gives the commit each release's dataset snapshot was taken at) and its
**clone** (`repos/<repo>/`). Whatever is missing is downloaded from the mined inputs on Zenodo
(10.5281/zenodo.24182007) by the same download cell as Notebooks 0–3, so every replication reviews the same
frozen clones at the same commits. A workspace that already has them, such as one that ran Quarry or
Notebooks 0–3, downloads nothing. The clones are only read, never changed.

**On Colab** the workspace is created in the session's temporary disk (about 2.5 GB of downloads), and only
the results go to Google Drive.

Each release's scores are saved as soon as it is done, so an interrupted run continues where it stopped (on
Colab, within the session, unless `KEEP_PROGRESS_ON_DRIVE = True`).

**Writes** `codescene_scores.parquet` (every test row's Code Health), `comparison_codescene.csv` and
`comparison_codescene.tex` to
`workspace_data/outputs/notebook_5_codescene/` (locally) or `MyDrive/notebook_5_codescene/outputs/` (Colab).

In [1]:
import io, os, re, sys, json, time, shutil, hashlib, tarfile, zipfile, tempfile, platform, subprocess, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
try:
    from sklearn.metrics import average_precision_score, roc_auc_score
except ImportError:
    raise SystemExit(f"scikit-learn is not installed for this kernel's Python ({sys.executable}). Install it with "
                     f"  %pip install scikit-learn  in a cell, then restart the kernel.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    COLAB = True
except ImportError:
    COLAB = False

KEEP_PROGRESS_ON_DRIVE = False   # Colab only. True: keep each release's scores on Drive, so a disconnect loses nothing

# The workspace the other notebooks share: QUARRY_HOME if set, otherwise workspace_data beside the codebase.
if COLAB:
    WORKSPACE = Path(os.environ.get("QUARRY_HOME", "/content/workspace_data"))      # temporary, like TMP below
else:
    WORKSPACE = Path(os.environ["QUARRY_HOME"]) if os.environ.get("QUARRY_HOME") \
        else Path.cwd().parent / "workspace_data"
if COLAB:
    OUT = Path("/content/drive/MyDrive/notebook_5_codescene/outputs")   # the results, on Google Drive
    TMP = Path("/content/notebook_5_codescene")                          # working files, gone with the session
    SCORES = (OUT.parent if KEEP_PROGRESS_ON_DRIVE else TMP) / "progress"
else:
    OUT = WORKSPACE / "outputs" / "notebook_5_codescene"
    TMP = WORKSPACE / "work" / "notebook_5_codescene"
    SCORES = TMP / "progress"
for d in (OUT, TMP, SCORES):
    d.mkdir(parents=True, exist_ok=True)

REPOS = ["checkstyle", "commons-lang", "dubbo", "junit-framework", "questdb"]
MINED_RECORD = "24182007"                               # Zenodo record of the mined inputs: release grids, clones
GITHUB_RAW = "https://raw.githubusercontent.com/Binamra00/refactoring-forecast-replication/HEAD"
if os.environ.get("NB5_REPOS"):                     # e.g. NB5_REPOS=commons-lang for a quick trial on one repo
    REPOS = [r for r in REPOS if r in os.environ["NB5_REPOS"].split(",")]

RELEASES_PER_REPO = None  # None = every test release (recommended); a number = that many per repo, spread evenly
WORKERS = 8               # cs review processes run side by side
print(f"workspace      {WORKSPACE}\noutputs        {OUT}\n"
      f"working files  {TMP}\nplatform       {platform.system()} {platform.machine()}")
if not shutil.which("git"):
    raise SystemExit("git is not installed or not on PATH")

workspace      E:\Thesis Project\Smell Ranker\workspace_data
outputs        E:\Thesis Project\Smell Ranker\workspace_data\outputs\notebook_5_codescene
working files  E:\Thesis Project\Smell Ranker\workspace_data\work\notebook_5_codescene
platform       Windows AMD64


## 1. The CodeScene CLI

Installed from CodeScene's own install script if `cs` is not found. The CLI needs a **Personal Access Token**
from your CodeScene account: create one at <https://codescene.io/users/me/pat>. The notebook looks for it in
this order, and asks for it only if it finds none:

1. the environment variable `CS_ACCESS_TOKEN`, if you set it before starting Jupyter;
2. on Colab, a secret named `CS_ACCESS_TOKEN` (the key icon in Colab's left bar, with notebook access switched on);
3. otherwise a hidden prompt, like a password prompt: paste the token and press Enter.

The token is kept only in this session's memory. It is never printed, written to a file or saved in the
notebook, so it is gone when the session ends and has to be given again next time (or stored as a Colab secret).

The cell then reviews a small Java file to check that the CLI runs and is signed in.

In [9]:
def find_cs():
    """The cs executable: on PATH, or where CodeScene's install scripts put it (Windows: AppData/Local/Programs/CodeScene)."""
    found = shutil.which("cs")
    win = Path(os.environ.get("USERPROFILE", Path.home())) / "AppData/Local/Programs/CodeScene"
    for p in [Path.home() / ".local/bin/cs", win / "cs.exe", *sorted(win.glob("cs*.exe"))]:
        if not found and p.exists():
            found = str(p)
    return found


CS = find_cs()
if CS is None:
    print("installing the CodeScene CLI ...")
    if platform.system() == "Windows":
        ps1 = TMP / "install-cs-tool.ps1"
        urllib.request.urlretrieve("https://downloads.codescene.io/enterprise/cli/install-cs-tool.ps1", ps1)
        subprocess.run(["powershell", "-ExecutionPolicy", "Bypass", "-File", str(ps1)], check=True)
    else:
        sh = TMP / "install-cs-tool.sh"
        urllib.request.urlretrieve("https://downloads.codescene.io/enterprise/cli/install-cs-tool.sh", sh)
        subprocess.run(["sh", str(sh), "-y"], check=True)
    CS = find_cs()
if CS is None:
    raise SystemExit("the CodeScene CLI did not install; see https://codescene.io/docs/cli/index.html")

def access_token():
    """The Personal Access Token: environment variable, then Colab secret, then a hidden prompt. Never printed."""
    if os.environ.get("CS_ACCESS_TOKEN", "").strip():
        return os.environ["CS_ACCESS_TOKEN"].strip(), "the CS_ACCESS_TOKEN environment variable"
    if COLAB:
        try:
            from google.colab import userdata
            token = (userdata.get("CS_ACCESS_TOKEN") or "").strip()
            if token:
                return token, "the Colab secret CS_ACCESS_TOKEN"
        except Exception:
            pass                               # no such secret, or notebook access not granted
    from getpass import getpass
    token = getpass("CodeScene Personal Access Token (from https://codescene.io/users/me/pat): ").strip()
    if not token:
        raise SystemExit("no token given")
    return token, "the prompt"


TOKEN, TOKEN_FROM = access_token()
ENV = {**os.environ, "CS_ACCESS_TOKEN": TOKEN, "CS_DISABLE_VERSION_CHECK": "1"}   # passed only to cs, in memory
print(subprocess.run([CS, "version"], capture_output=True, text=True, env=ENV).stdout.strip())
print(f"token from {TOKEN_FROM} (not shown)")

PROBE = """public class Probe {
    int f(int a, int b) {
        if (a > b && b > 0) { for (int i = 0; i < a; i++) { if (i % 2 == 0) { b += i; } else { b -= i; } } }
        return b;
    }
}
"""


def review(paths, cwd):
    """CodeScene Code Health for the given files (paths relative to `cwd`): {path: score or None}."""
    r = subprocess.run([CS, "review", *paths, "--output-format", "jsonl"], cwd=cwd, capture_output=True,
                       text=True, env=ENV)
    lines = [json.loads(l) for l in r.stdout.splitlines() if l.strip().startswith("{")]
    if r.returncode not in (0, 1) and not lines:
        raise RuntimeError((r.stderr or r.stdout).replace(TOKEN, "<token>").strip()[:500])
    if len(paths) == 1 and len(lines) == 1:
        return {paths[0]: score_of(lines[0])}
    out = {}
    for obj in lines:
        name = next((obj[k] for k in ("name", "file-name", "file_name", "filename", "file", "path") if k in obj), None)
        if name is not None:
            out[norm(name)] = score_of(obj)
    return {p: out.get(norm(p)) for p in paths}


def score_of(obj):
    s = obj.get("score")
    return float(s) if isinstance(s, (int, float)) else None


def norm(p):
    return str(p).replace("\\", "/").lstrip("./")


probe_dir = TMP / "probe"
probe_dir.mkdir(exist_ok=True)
(probe_dir / "Probe.java").write_text(PROBE, encoding="utf-8")
raw = subprocess.run([CS, "review", "Probe.java", "--output-format", "json"], cwd=probe_dir, capture_output=True,
                     text=True, env=ENV)
print("cs review on a test file:", (raw.stdout or raw.stderr).replace(TOKEN, "<token>").strip()[:600])
try:
    probe_score = score_of(json.loads(raw.stdout))
except json.JSONDecodeError:
    probe_score = None
if probe_score is None:
    raise SystemExit("cs review did not return a Code Health score; CodeScene's message is shown above. If it "
                     "asks for a token, check the token at https://codescene.io/users/me/pat and run this cell again.")
(probe_dir / "Probe2.java").write_text(PROBE.replace("Probe", "Probe2"), encoding="utf-8")
BATCH = review(["Probe.java", "Probe2.java"], probe_dir) == {"Probe.java": probe_score, "Probe2.java": probe_score}
print(f"Code Health of the test file {probe_score}; files reviewed "
      f"{'in batches' if BATCH else 'one per call (the batch output names no files)'}")

CodeScene Personal Access Token (from https://codescene.io/users/me/pat):  ········


cs version 1.0.58 (b2580d71be8fbfd6a496d7a007062c0c81cd2b60)
Built on Windows Server 2025 amd64 at 2026-11-09
token from the prompt (not shown)
cs review on a test file: {"score":10.0,"review":[],"path":"Probe.java"}
Code Health of the test file 10.0; files reviewed in batches


## 2. The test rows and the model's forecasts

Run 3's forecasts (`results/run3/outputs/forecasts.parquet` in the replication repo, written by Notebook 4) hold
every test row with its repo, release, file, label and the model's probability, so the dataset itself is not
needed here. CodeScene scores every test release by default; a few releases per repo is quicker but noisier.

In [3]:
# the copy in this repo's results/run3/outputs/ when the notebook runs inside a checkout of the replication repo,
# otherwise the same file downloaded from GitHub into the working folder
here = [Path.cwd() / "results/run3/outputs/forecasts.parquet", Path.cwd().parent / "results/run3/outputs/forecasts.parquet",
        TMP / "forecasts_run3.parquet"]
FORECASTS = next((p for p in here if p.exists()), None)
if FORECASTS is None:
    FORECASTS = TMP / "forecasts_run3.parquet"
    urllib.request.urlretrieve(f"{GITHUB_RAW}/results/run3/outputs/forecasts.parquet", FORECASTS)
F = pd.read_parquet(FORECASTS)
TEST = F[F.part == "test"][["row_id", "repo", "release", "path", "label", "probability"]].reset_index(drop=True)
ap_all = average_precision_score(TEST.label, TEST.probability)
print(f"{FORECASTS}\n{len(TEST):,} test rows, model AP {ap_all:.4f} (Chapter 5: 0.079)")
if abs(ap_all - 0.0788) > 0.0005:
    raise SystemExit("these are not run 3's forecasts")


def release_order(repo):
    """Test releases of a repo in time order, from their commits' dates (section 3)."""
    rels = TEST.release[TEST.repo == repo].unique()
    return sorted(rels, key=lambda t: commit_time(repo, SHA[repo][t]))


def pick(releases, n):
    if n is None or n >= len(releases):
        return list(releases)
    return [releases[i] for i in np.unique(np.linspace(0, len(releases) - 1, n).round().astype(int))]

E:\Thesis Project\Smell Ranker\workspace_data\work\notebook_5_codescene\forecasts_run3.parquet
200,438 test rows, model AP 0.0788 (Chapter 5: 0.079)


## 3. The release grids and clones, and the commit of each test release

Only **test-split releases** are scored: the releases of `part == "test"` in run 3's forecasts, the ones the
fine-tuned model was evaluated on. Within a release, only the files that are test rows are reviewed.

For each repo this cell downloads only what is missing from the workspace, as Notebooks 0–3 do:
- the release grid, from the repo's zip of mined inputs, checked against the record's `SHA256SUMS.txt` and the
  zip's `MANIFEST_<repo>.json` (only the grid is unpacked);
- the clone, from the repo's git bundle, checked the same way and restored in `repos/<repo>/`.

Each release is then read at the commit its dataset snapshot was taken at, the grid's `snapshot_sha`.

In [4]:
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


def download(name, dest, sums):
    """One file of the record, checked against SHA256SUMS.txt before it is kept."""
    tmp = dest.with_name(dest.name + ".part")
    print(f"   downloading {name}")
    urllib.request.urlretrieve(f"https://zenodo.org/records/{MINED_RECORD}/files/{name}?download=1", tmp)
    if sha256_of(tmp) != sums[name]:
        tmp.unlink()
        raise SystemExit(f"{name}: SHA-256 differs from the record's SHA256SUMS.txt; download it again")
    tmp.replace(dest)


def fetch_mined(repos):
    downloads = WORKSPACE / "downloads"
    sums = None
    for repo in repos:
        grid, clone = WORKSPACE / "versions" / f"rel_hist_{repo}.json", WORKSPACE / "repos" / repo
        need_grid, need_clone = not grid.exists(), not (clone / ".git").exists()
        if not (need_grid or need_clone):
            print(f"{repo:<18}release grid and clone already in {WORKSPACE}")
            continue
        downloads.mkdir(parents=True, exist_ok=True)
        if sums is None:
            urllib.request.urlretrieve(f"https://zenodo.org/records/{MINED_RECORD}/files/SHA256SUMS.txt?download=1",
                                       downloads / "SHA256SUMS.txt")
            sums = {n: h for h, n in (l.split() for l in
                    (downloads / "SHA256SUMS.txt").read_text(encoding="utf-8").splitlines() if l.strip())}
        if need_grid:
            z = downloads / f"{repo}_mined.zip"
            download(z.name, z, sums)
            with zipfile.ZipFile(z) as zf:
                manifest = json.loads(zf.read(f"MANIFEST_{repo}.json"))
                f = next(f for f in manifest["files"] if f["file"].endswith(f"rel_hist_{repo}.json"))
                grid.parent.mkdir(parents=True, exist_ok=True)
                grid.write_bytes(zf.read(f["file"]))
                if sha256_of(grid) != f["sha256"]:
                    raise SystemExit(f"{f['file']}: SHA-256 differs from MANIFEST_{repo}.json")
            z.unlink()
            print(f"{repo:<18}release grid unpacked and checked")
        if need_clone:
            b = downloads / f"{repo}.bundle"
            download(b.name, b, sums)
            clone.mkdir(parents=True, exist_ok=True)
            for args in (["clone", "--mirror", "--quiet", str(b.resolve()), str(clone / ".git")],
                         ["-C", str(clone), "config", "--bool", "core.bare", "false"],
                         ["-C", str(clone), "-c", "core.longpaths=true", "reset", "--hard", "--quiet"]):
                r = subprocess.run(["git", *args], capture_output=True, text=True)
                if r.returncode != 0:
                    raise SystemExit(f"git {' '.join(args[:2])} failed for {repo}: {r.stderr.strip()[:300]}")
            b.unlink()
            print(f"{repo:<18}clone restored from {b.name}")


fetch_mined(REPOS)
CLONE = {repo: WORKSPACE / "repos" / repo for repo in REPOS}         # read only: nothing in them is changed


def git(repo, *args, binary=False):
    r = subprocess.run(["git", *args], cwd=CLONE[repo], capture_output=True, text=not binary)
    if r.returncode:
        err = r.stderr if not binary else r.stderr.decode("utf-8", "replace")
        raise RuntimeError(f"git {' '.join(args[:4])}: {err.strip()[:300]}")
    return r.stdout


def commit_time(repo, sha):
    return int(git(repo, "log", "-1", "--format=%ct", sha).strip())


SHA = {}
for repo in REPOS:
    grid_file = WORKSPACE / "versions" / f"rel_hist_{repo}.json"
    grid = {e.get("tag"): e["snapshot_sha"] for e in json.loads(grid_file.read_text(encoding="utf-8"))
            .get("entries", []) if e.get("snapshot_sha")}
    SHA[repo], from_tag, differs = {}, 0, 0
    for tag in TEST.release[TEST.repo == repo].unique():
        tag_sha = None
        try:
            tag_sha = git(repo, "rev-parse", f"refs/tags/{tag}^{{commit}}").strip()
        except RuntimeError:
            pass
        sha = grid.get(tag) or tag_sha
        if sha is None:
            raise SystemExit(f"{repo}: release {tag} is neither in the release grid nor a tag of the clone")
        from_tag += tag not in grid
        differs += bool(grid.get(tag) and tag_sha and grid[tag] != tag_sha)
        SHA[repo][tag] = sha
    n = len(SHA[repo])
    print(f"{repo:<17}clone {CLONE[repo]}\n{'':<17}{n - from_tag} of {n} test releases from the release grid, "
          f"{from_tag} from git tags" + (f"; {differs} snapshot commits differ from their tag's" if differs else ""))


CHOSEN = {repo: pick(release_order(repo), RELEASES_PER_REPO) for repo in REPOS}
# guard: only test-split releases, the ones the fine-tuned model was evaluated on
not_test = {(r, t) for r, rels in CHOSEN.items() for t in rels} - set(zip(TEST.repo, TEST.release))
assert not not_test, f"not test releases: {not_test}"
for repo, rels in CHOSEN.items():
    n = int(TEST[(TEST.repo == repo) & TEST.release.isin(rels)].shape[0])
    print(f"{repo:<17}{len(rels):>3} of {TEST.release[TEST.repo == repo].nunique():>2} test releases, {n:>6,} files: "
          f"{', '.join(rels)}")

checkstyle        release grid and clone already in E:\Thesis Project\Smell Ranker\workspace_data
commons-lang      release grid and clone already in E:\Thesis Project\Smell Ranker\workspace_data
dubbo             release grid and clone already in E:\Thesis Project\Smell Ranker\workspace_data
junit-framework   release grid and clone already in E:\Thesis Project\Smell Ranker\workspace_data
questdb           release grid and clone already in E:\Thesis Project\Smell Ranker\workspace_data
checkstyle       clone E:\Thesis Project\Smell Ranker\workspace_data\repos\checkstyle
                 60 of 60 test releases from the release grid, 0 from git tags
commons-lang     clone E:\Thesis Project\Smell Ranker\workspace_data\repos\commons-lang
                 8 of 8 test releases from the release grid, 0 from git tags
dubbo            clone E:\Thesis Project\Smell Ranker\workspace_data\repos\dubbo
                 32 of 32 test releases from the release grid, 0 from git tags
junit-framework  clo

## 4. Code Health at each release

The release's test files are copied out of git (`git archive` of the release's commit, which only reads the
clone) into a temporary folder, each under a short numbered folder with its own file name, so that no path
passes Windows' 260-character limit. Code Health depends on a file's content, not its location. Every file is
reviewed there by `cs review`, and the folder is deleted afterwards. The first release is timed and the full run estimated before going on. A release's scores are saved when it finishes, and a release
already saved is not reviewed again.

In [5]:
def review_safely(paths, cwd):
    """review(), falling back to one file per call if a batch fails; a file that still fails gets no score."""
    try:
        return review(paths, cwd)
    except Exception as e:
        if len(paths) > 1:
            return {k: v for p in paths for k, v in review_safely([p], cwd).items()}
        FAILED.append((str(cwd), paths[0], str(e)[:200]))
        return {paths[0]: None}


FAILED = []


def export(repo, sha, wanted, dest):
    """Copy the release's files in `wanted` into `dest`, each as <number>/<its file name>, and return
    {repo path: short path}. Code Health is read from a file's content, not from where it sits, and the short
    paths keep every file well under Windows' 260-character path limit (some Dubbo paths alone are 230).
    CodeScene's settings folder, if the repo has one, is copied as it is."""
    specs = ["*.java"] + ([".codescene"] if git(repo, "ls-tree", "--name-only", sha, ".codescene").strip() else [])
    data = git(repo, "archive", "--format=tar", sha, "--", *specs, binary=True)
    short = {}
    with tarfile.open(fileobj=io.BytesIO(data)) as tar:
        for m in tar.getmembers():
            if not m.isfile():
                continue
            if m.name in wanted:
                rel = f"{len(short):05d}/{Path(m.name).name}"
                short[m.name] = rel
            elif m.name.startswith(".codescene/"):
                rel = m.name
            else:
                continue
            target = dest / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(tar.extractfile(m).read())
    return short


def score_release(repo, tag):
    out = SCORES / repo / (re.sub(r"[^\w.-]", "_", tag) + ".parquet")
    if out.exists():
        return pd.read_parquet(out, columns=["row_id", "in_checkout", "code_health"]), 0.0
    rows = TEST[(TEST.repo == repo) & (TEST.release == tag)]
    wt = Path(tempfile.mkdtemp(prefix="cs_"))                # short path: long Java paths fit on Windows too
    t0 = time.time()
    try:
        short = export(repo, SHA[repo][tag], set(rows.path), wt)
        paths = list(short.values())
        chunks = [paths[i:i + 25] for i in range(0, len(paths), 25)] if BATCH else [[p] for p in paths]
        by_short = {}
        with ThreadPoolExecutor(WORKERS) as ex:
            for res in ex.map(lambda c: review_safely(c, wt), chunks):
                by_short.update(res)
        health = {p: by_short.get(sp) for p, sp in short.items()}
    finally:
        shutil.rmtree(wt, ignore_errors=True)
    S = rows[["row_id"]].copy()
    S["in_checkout"] = rows.path.map(lambda p: p in health).to_numpy()
    S["code_health"] = rows.path.map(health).astype(float).to_numpy()
    out.parent.mkdir(parents=True, exist_ok=True)
    S.to_parquet(out, index=False)
    return S, time.time() - t0


def recheck_saved(n=50):
    """If releases were saved by an earlier run, review n of their files again now and compare the scores, so a
    change in how files are reviewed (or in CodeScene) cannot mix two kinds of score in one comparison."""
    saved = [(r, t) for r, rels in CHOSEN.items() for t in rels
             if (SCORES / r / (re.sub(r"[^\w.-]", "_", t) + ".parquet")).exists()]
    if not saved:
        return
    repo, tag = saved[0]
    S = pd.read_parquet(SCORES / repo / (re.sub(r"[^\w.-]", "_", tag) + ".parquet")).dropna(subset=["code_health"])
    S = S.merge(TEST[["row_id", "path"]], on="row_id").head(n)
    wt = Path(tempfile.mkdtemp(prefix="cs_"))
    try:
        short = export(repo, SHA[repo][tag], set(S.path), wt)
        now = review(list(short.values()), wt) if BATCH else {v: review([v], wt)[v] for v in short.values()}
    finally:
        shutil.rmtree(wt, ignore_errors=True)
    again = S.path.map(lambda p: now.get(short.get(p)))
    same = int((again.round(2) == S.code_health.round(2)).sum())
    print(f"{len(saved)} releases already saved; {same} of {len(S)} files of {repo} {tag} score the same when "
          f"reviewed again now")
    if same < len(S):
        raise SystemExit("saved scores differ from today's reviews: delete the progress folder "
                         f"({SCORES}) and run this cell again, so every release is scored the same way")


recheck_saved()
todo = [(r, t) for r, rels in CHOSEN.items() for t in rels]
parts, t_start, estimated = [], time.time(), False
for i, (repo, tag) in enumerate(todo):
    S, secs = score_release(repo, tag)
    parts.append(S)
    n = len(S)
    msg = f"{i + 1:>3}/{len(todo)}  {repo:<17}{tag:<28}{n:>6,} files"
    if secs:
        msg += f"  {secs / 60:5.1f} min ({secs / max(n, 1):.2f} s per file)"
        if not estimated:
            estimated = True
            left = sum(int(((TEST.repo == r) & (TEST.release == t)).sum()) for r, t in todo[i + 1:])
            msg += f"\n     estimate for the remaining {left:,} files: {left * secs / max(n, 1) / 3600:.1f} h"
    print(msg)
if FAILED:
    print(f"{len(FAILED)} files could not be reviewed, for example: {FAILED[0]}")
CS_SCORES = pd.concat(parts, ignore_index=True)[["row_id", "in_checkout", "code_health"]]
CS_SCORES.to_parquet(OUT / "codescene_scores.parquet", index=False)
print(f"\nreviewed in {(time.time() - t_start) / 60:.0f} min")

148 releases already saved; 50 of 50 files of checkstyle checkstyle-10.9.2 score the same when reviewed again now
  1/148  checkstyle       checkstyle-10.9.2              399 files
  2/148  checkstyle       checkstyle-10.9.3              400 files
  3/148  checkstyle       checkstyle-10.10.0             401 files
  4/148  checkstyle       checkstyle-10.11.0             401 files
  5/148  checkstyle       checkstyle-10.12.0             401 files
  6/148  checkstyle       checkstyle-10.12.1             401 files
  7/148  checkstyle       checkstyle-10.12.2             406 files
  8/148  checkstyle       checkstyle-10.12.3             409 files
  9/148  checkstyle       checkstyle-10.12.4             411 files
 10/148  checkstyle       checkstyle-10.12.5             411 files
 11/148  checkstyle       checkstyle-10.12.6             411 files
 12/148  checkstyle       checkstyle-10.12.7             411 files
 13/148  checkstyle       checkstyle-10.13.0             412 files
 14/148  checks

## 5. The comparison

For each repo: the model's and Code Health's AP, lift (AP over the base rate) and ROC-AUC on that repo's test
rows, and by how many percent the model's AP is above (or below) Code Health's,
100 × (AP_model − AP_Code Health) / AP_Code Health. The **mean over repos** averages each method's five values
and compares the two means; each repo counts once. The model's per-repo values are the per-repo results of
Chapter 5, recomputed from run 3's forecasts.

A file CodeScene gives no score counts as healthy (10), and the comparison is repeated on the scored files alone.

In [6]:
D = TEST.merge(CS_SCORES, on="row_id", how="inner")
# guard: CodeScene scored exactly the test rows of the chosen releases, no more and no fewer
want = TEST[[(r, t) in {(r, t) for r, rels in CHOSEN.items() for t in rels} for r, t in zip(TEST.repo, TEST.release)]]
assert set(D.row_id) == set(want.row_id), "the scored rows differ from the test rows of the chosen releases"
print(f"{D.release.nunique()} test releases of {D.repo.nunique()} repos, {len(D):,} test rows "
      f"(the same rows the model was evaluated on)")
D["scored"] = D.code_health.notna()
D["code_health_score"] = 10 - D.code_health.fillna(10.0).clip(1, 10)       # unhealthiest first
print(f"{int((~D.scored).sum()):,} rows have no Code Health score and count as healthy; refactored "
      f"{D.label[~D.scored].mean():.1%} of them, against {D.label[D.scored].mean():.1%} of scored files")


def compare(D, scopes):
    rows = []
    for name in scopes:
        g = D[D.repo == name]
        y = g.label.to_numpy(int)
        r = {"scope": name, "rows": len(g), "releases": g.release.nunique(), "base rate": y.mean()}
        for key, col in (("model", "probability"), ("Code Health", "code_health_score")):
            r[f"AP {key}"] = average_precision_score(y, g[col])
            r[f"lift {key}"] = r[f"AP {key}"] / y.mean()
            r[f"ROC-AUC {key}"] = roc_auc_score(y, g[col])
        rows.append(r)
    T = pd.DataFrame(rows)
    per = T[T.scope.isin(REPOS)]
    mean = per.drop(columns=["scope", "rows", "releases"]).mean().to_dict()
    mean.update(scope="mean over repos", rows=int(per.rows.sum()), releases=int(per.releases.sum()))
    T = pd.concat([T, pd.DataFrame([mean])], ignore_index=True)
    T["model vs Code Health, AP %"] = 100 * (T["AP model"] - T["AP Code Health"]) / T["AP Code Health"]
    return T


CMP = compare(D, REPOS)
CMP.to_csv(OUT / "comparison_codescene.csv", index=False)
show = CMP[["scope", "rows", "base rate", "AP model", "AP Code Health", "model vs Code Health, AP %",
            "ROC-AUC model", "ROC-AUC Code Health"]]
print("\n" + show.round(3).to_string(index=False))

per = CMP[CMP.scope.isin(REPOS)]
print(f"\nthe model's AP is higher in {int((per['AP model'] > per['AP Code Health']).sum())} of {len(per)} repos, "
      f"its ROC-AUC in {int((per['ROC-AUC model'] > per['ROC-AUC Code Health']).sum())} of {len(per)}")

ONLY = compare(D[D.scored], REPOS)
m = ONLY[ONLY.scope == "mean over repos"].iloc[0]
print(f"scored files only, mean over repos: AP model {m['AP model']:.3f}, Code Health {m['AP Code Health']:.3f} "
      f"({m['model vs Code Health, AP %']:+.1f}%)")

148 test releases of 5 repos, 200,438 test rows (the same rows the model was evaluated on)
25,212 rows have no Code Health score and count as healthy; refactored 1.2% of them, against 3.5% of scored files

          scope   rows  base rate  AP model  AP Code Health  model vs Code Health, AP %  ROC-AUC model  ROC-AUC Code Health
     checkstyle  25323      0.017     0.060           0.043                      38.421          0.756                0.693
   commons-lang   1820      0.146     0.293           0.278                       5.494          0.693                0.654
          dubbo  63649      0.026     0.052           0.043                      18.880          0.673                0.619
junit-framework  10577      0.028     0.069           0.039                      75.752          0.755                0.543
        questdb  99069      0.038     0.130           0.109                      19.420          0.756                0.652
mean over repos 200438      0.051     0.120       

## 6. The table for Chapter 5

The per-repo rows and their mean, as in `comparison_codescene.csv`.

In [7]:
NAME = {"checkstyle": "Checkstyle", "commons-lang": "Commons Lang", "dubbo": "Dubbo",
        "junit-framework": "JUnit", "questdb": "QuestDB", "mean over repos": "Mean over repos"}
tex = [r"\begin{tabular}{@{}lrrrrrr@{}}", r"\toprule",
       r" & & \multicolumn{3}{c}{AP} & \multicolumn{2}{c}{ROC-AUC} \\",
       r"\cmidrule(lr){3-5} \cmidrule(l){6-7}",
       r"Repo & Base rate & Model & Code Health & Model vs.\ Code Health & Model & Code Health \\", r"\midrule"]
for _, r in CMP.iterrows():
    if r.scope == "mean over repos":
        tex.append(r"\midrule")
    tex.append(f"{NAME[r.scope]} & {r['base rate']:.3f} & {r['AP model']:.3f} & {r['AP Code Health']:.3f} & "
               f"{r['model vs Code Health, AP %']:+.0f}\\% & {r['ROC-AUC model']:.3f} & {r['ROC-AUC Code Health']:.3f} \\\\")
tex += [r"\bottomrule", r"\end{tabular}"]
(OUT / "comparison_codescene.tex").write_text("\n".join(tex) + "\n", encoding="utf-8")
print("\n".join(tex))
print(f"\nwrote codescene_scores.parquet, comparison_codescene.csv and comparison_codescene.tex to {OUT}")

\begin{tabular}{@{}lrrrrrr@{}}
\toprule
 & & \multicolumn{3}{c}{AP} & \multicolumn{2}{c}{ROC-AUC} \\
\cmidrule(lr){3-5} \cmidrule(l){6-7}
Repo & Base rate & Model & Code Health & Model vs.\ Code Health & Model & Code Health \\
\midrule
Checkstyle & 0.017 & 0.060 & 0.043 & +38\% & 0.756 & 0.693 \\
Commons Lang & 0.146 & 0.293 & 0.278 & +5\% & 0.693 & 0.654 \\
Dubbo & 0.026 & 0.052 & 0.043 & +19\% & 0.673 & 0.619 \\
JUnit & 0.028 & 0.069 & 0.039 & +76\% & 0.755 & 0.543 \\
QuestDB & 0.038 & 0.130 & 0.109 & +19\% & 0.756 & 0.652 \\
\midrule
Mean over repos & 0.051 & 0.120 & 0.102 & +18\% & 0.727 & 0.632 \\
\bottomrule
\end{tabular}

wrote codescene_scores.parquet, comparison_codescene.csv and comparison_codescene.tex to E:\Thesis Project\Smell Ranker\workspace_data\outputs\notebook_5_codescene
